# Method: Ridge Regression

#### Algorithm Reference

T. Hastie, R. Tibshirani, and J. Friedman, *The Elements of Statistical Learning: Data Mining, Inference, and Prediction*, 2nd ed. New York, NY, USA: Springer, 2009, sec. 3.4.1.

### I. Imports and Reads

#### Imports

In [11]:
import numpy as np
import csv 
import EvaluationMetrics as em

#### Reads

In [12]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data)
    test_out = np.array(output)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data)
    tune_out = np.array(output)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data)
    train_out = np.array(output)

train_in

array([['5', '0.016057125812515255', '-13.062962962962963',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ['5', '0.016057125812515255', '-12.596296296296295',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ['5', '0.016057125812515255', '-13.596296296296295',
        '38.190444444444445', '37.596296296296295', '37.00214814814815'],
       ...,
       ['2', '0.02745569032449328', '0.5259574468084978',
        '37.229347826086965', '37.425000000000004', '36.258539325842705'],
       ['2', '0.02745569032449328', '-4.254042553191503',
        '37.229347826086965', '37.425000000000004', '36.258539325842705'],
       ['2', '0.02745569032449328', '0.07595744680849492',
        '37.229347826086965', '37.425000000000004', '36.258539325842705']],
      shape=(3174, 6), dtype='<U22')

### II. Setup

#### Ridge Regression Functions

In [13]:
LAMB = 1  # hyperparameter, the lambda value

def compute_train_stats(X, y):
    x_mean = np.mean(X, axis=0) 			# one mean per column
    x_std = np.std(X, axis=0)				# one std per column
    y_mean = np.mean(y)
    return x_mean, x_std, y_mean

def standardize(X, x_mean, x_std):
    Xs = (X - x_mean) / x_std
    return Xs

def solve_coefficients(lamb, Xs, yc):
    p = Xs.shape[1]
    A = Xs.T @ Xs + lamb * np.eye(p)	  	# XTX + lambI
    b = Xs.T @ yc  							# XTy
    coeff = np.linalg.solve(A, b)
    return coeff

def fit(lamb, X, y):
    # get the training stats
    x_mean, x_std, y_mean = compute_train_stats(X, y)

	# standardize X and center y
    Xs = standardize(X, x_mean, x_std)
    yc = y - y_mean

	# solve for coeffs
    coeff = solve_coefficients(lamb, Xs, yc)
    intercept = y_mean

    return {
        'coeff': coeff,
        'intercept': intercept,
        'x_mean': x_mean,
        'x_std': x_std,
    }

def predict(model, X):
    # standardize X using the training statistics
    Xs = standardize(X, model['x_mean'], model['x_std'])

    # apply the intercept and coefficients
    y_pred = model['intercept'] + Xs @ model['coeff']
    return y_pred

### III. Training

#### Perform Ridge Regression

In [14]:
# convert CSV strings to numbers
X_train = train_in.astype(float)
y_train = train_out.astype(float)
X_tune = tune_in.astype(float)
y_tune = tune_out.astype(float)

# train, then predict
model = fit(LAMB, X_train, y_train)
y_pred = predict(model, X_tune)

#### Display Ridge Regression Predictions

In [15]:
provinces = ['Metropolitan Manila', 'Nueva Ecija', 'Iloilo', 'South Cotabato', 'Basilan', 'Maguindanao', 'Ifugao', 'Kalinga', 'Ilocos Norte', 'Ilocos Sur', 'Cagayan', 'Isabela',
 'Quirino', 'Bataan', 'Bulacan', 'Batangas', 'Cavite', 'Laguna', 'Quezon', 'Rizal', 'Marinduque', 'Occidental Mindoro', 'Oriental Mindoro', 'Palawan', 'Romblon', 'Zamboanga del Sur',
 'Albay', 'Catanduanes', 'Masbate', 'Sorsogon', 'Aklan', 'Guimaras', 'Negros Occidental', 'Bohol', 'Siquijor', 'Biliran', 'Leyte', 'Northern Samar', 'Samar', 'Southern Leyte', 'Bukidnon',
 'Camiguin', 'Lanao del Norte', 'Misamis Occidental', 'Misamis Oriental', 'Davao Oriental', 'Davao del Norte', 'North Cotabato', 'Sarangani', 'Agusan del Norte', 'Surigao del Norte',
 'Surigao del Sur']

labels = []
with open('../dataset/wfp_food_prices_phl.csv', 'r') as f:
    read = csv.reader(f)
    for x in read:
        if x[0][:4] == '2025' and x[8] == 'Rice (regular, milled)' and x[12] == 'Retail' and x[2] in provinces:
            labels.append({
                'year' : int(x[0][:4]),
                'month' : int(x[0][5:7]),
                'province' : x[2],
                'price' : float(x[14]),
            })

# Stop if the labels do not line up with the 2025 rows
assert len(labels) == len(y_tune)
for i in range(len(labels)):
    assert labels[i]['month'] == X_tune[i, 0]
    assert labels[i]['price'] == y_tune[i]

# Print each prediction next to the correct answer
print(f"{'Month':>5}  {'Market':<22} {'Predicted':>10} {'Actual':>10} {'Error':>8}")
for label, pred, actual in zip(labels, y_pred, y_tune):
    print(f"{label['month']:>5}  {label['province']:<22} {pred:>10.2f} {actual:>10.2f} {pred - actual:>8.2f}")

Month  Market                  Predicted     Actual    Error
    1  Basilan                     53.52      50.39     3.13
    1  Maguindanao                 49.64      47.75     1.89
    1  Ifugao                      50.27      48.83     1.44
    1  Kalinga                     51.47      50.88     0.59
    1  Metropolitan Manila         47.68      44.52     3.16
    1  Ilocos Norte                48.64      43.63     5.01
    1  Ilocos Sur                  48.58      46.23     2.35
    1  Cagayan                     46.37      42.86     3.51
    1  Isabela                     48.46      46.38     2.08
    1  Quirino                     45.95      40.53     5.42
    1  Bataan                      48.48      46.00     2.48
    1  Bulacan                     49.20      44.62     4.58
    1  Bulacan                     49.20      46.73     2.47
    1  Nueva Ecija                 47.21      44.67     2.54
    1  Batangas                    54.93      49.93     5.00
    1  Cavite           

#### Evaluation Metrics on Initial Results

In [ ]:
print(f"Pre-tuned results, lambda = {LAMB}")
print(f"MAE:  	₱{em.mae(y_tune, y_pred):.4f}/kg")
print(f"RMSE: 	₱{em.rmse(y_tune, y_pred):.4f}/kg")
print(f"MAPE: 	{em.mape(y_tune, y_pred):.4f}%")
print(f"R2:   	{em.r_squared(y_tune, y_pred):.4f}")

Pre-tuned results, lambda = 1
MAE:  	₱1.6823/kg
RMSE: 	₱2.1649/kg
MAPE: 	4.0182%
R2:   	0.7703


#### Hyperparameter Tuning

In [17]:
import time

LAMBS = [0, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]

results = []
for lamb in LAMBS:
	# time the fitting on the training data
	start = time.perf_counter()
	m = fit(lamb, X_train, y_train)
	train_time = time.perf_counter() - start

	# get score of the current lamb on the validation set
	pred = predict(m, X_tune)
	results.append({
		'lambda': lamb,
		'mae': em.mae(y_tune, pred),
        'rmse': em.rmse(y_tune, pred),
        'mape': em.mape(y_tune, pred),
        'r2': em.r_squared(y_tune, pred),
        'time': train_time,
	})

# print results of each lambda tested
print(f"{'lambda':>9} {'MAE':>12} {'RMSE':>12} {'MAPE %':>12} {'R2':>12} {'time (s)':>10}")
for r in results:
    print(f"{r['lambda']:>9g} {r['mae']:>12.8f} {r['rmse']:>12.8f} {r['mape']:>12.8f} {r['r2']:>12.8f} {r['time']:>10.5f}")

   lambda          MAE         RMSE       MAPE %           R2   time (s)
        0   1.68191187   2.16460249   4.01712224   0.77037262    0.00058
    1e-05   1.68191188   2.16460250   4.01712225   0.77037262    0.00030
   0.0001   1.68191191   2.16460253   4.01712235   0.77037261    0.00030
    0.001   1.68191228   2.16460282   4.01712332   0.77037255    0.00028
     0.01   1.68191594   2.16460573   4.01713309   0.77037193    0.00030
      0.1   1.68195255   2.16463488   4.01723070   0.77036575    0.00083
        1   1.68231735   2.16492530   4.01820350   0.77030412    0.00051
       10   1.68587627   2.16772713   4.02771051   0.76970920    0.00055
      100   1.71233198   2.18838871   4.10020549   0.76529827    0.00049
     1000   1.83899921   2.28920212   4.45413826   0.74317598    0.00032
    10000   2.64336182   3.25333437   6.32154489   0.48128982    0.00027
   100000   3.48634030   4.35219381   8.19342981   0.07170968    0.00028


#### Best λ

In [ ]:
# the lambda with the lowest validation MAE
best = min(results, key=lambda r: r['mae'])
BEST_LAMB = best['lambda']

print(f"Best tuned results, lambda = {BEST_LAMB}")
print(f"MAE:  \t₱{best['mae']:.4f}/kg")
print(f"RMSE: \t₱{best['rmse']:.4f}/kg")
print(f"MAPE: \t{best['mape']:.4f}%")
print(f"R2:   \t{best['r2']:.4f}")

Best tuned results, lambda = 0
MAE:  	₱1.6819/kg
RMSE: 	₱2.1646/kg
MAPE: 	4.0171%
R2:   	0.7704


#### Final Model

In [ ]:
# final model combines the training and validation data (2020–2025)
# vstack puts the validation rows below the training rows
X_final = np.vstack([X_train, X_tune])
# concatenate joins the two lists of target prices in the same order
y_final = np.concatenate([y_train, y_tune])

# refit with the chosen lambda on validation set, and time
start = time.perf_counter()
final_model = fit(BEST_LAMB, X_final, y_final)
final_train_time = time.perf_counter() - start

print(f"Final model\nlambda: {BEST_LAMB}, training time: {final_train_time:.5f} s")

Final model
lambda: 0, training time: 0.00100 s


### IV. Test Model on the Test Dataset

In [21]:
# convert CSV strings to numbers
X_test = test_in.astype(float)
y_test = test_out.astype(float)

# predict the test set with the tuned model (fitted on the training + validation set)
y_pred_test = predict(final_model, X_test)

print(f"Test results, lambda = {BEST_LAMB}")
print(f"MAE:  \t₱{em.mae(y_test, y_pred_test):.4f}/kg")
print(f"RMSE: \t₱{em.rmse(y_test, y_pred_test):.4f}/kg")
print(f"MAPE: \t{em.mape(y_test, y_pred_test):.4f}%")
print(f"R2:   \t{em.r_squared(y_test, y_pred_test):.4f}")

Test results, lambda = 0
MAE:  	₱2.0795/kg
RMSE: 	₱2.6998/kg
MAPE: 	4.6300%
R2:   	0.6938
